In [1]:
import numpy as np
import pandas as pd
import os

input_dir = '/kaggle/input/playground-series-s5e10'

In [2]:
def ingest(input_dir):
    target = 'accident_risk'

    train_csv = pd.read_csv(os.path.join(input_dir, 'train.csv'))
    X_train_df = train_csv.drop(columns=['id', target])
    
    categories = X_train_df.select_dtypes(exclude=np.number).columns.tolist()
    
    for col in categories:
        X_train_df[col] = X_train_df[col].astype('category')

    y_train_df = train_csv[target]
    
    return X_train_df, y_train_df

X_train_df, y_train_df = ingest(input_dir)
print(X_train_df.shape, y_train_df.shape)
X_train_df

(517754, 12) (517754,)


,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents
0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1
1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0
2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2
3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1
4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1
...,...,...,...,...,...,...,...,...,...,...,...,...
517749,highway,4,0.10,70,daylight,foggy,True,True,afternoon,False,False,2
517750,rural,4,0.47,35,daylight,rainy,True,True,morning,False,False,1
517751,urban,4,0.62,25,daylight,foggy,False,False,afternoon,False,True,0
517752,highway,3,0.63,25,night,clear,True,False,afternoon,True,True,3


In [3]:
from sklearn.model_selection import train_test_split
import xgboost as xgb

X_train_set, X_test_set, y_train_set, y_test_set = train_test_split(X_train_df, y_train_df, test_size=0.2, random_state=42)

dtrain_reg = xgb.DMatrix(X_train_set, y_train_set, enable_categorical=True)
dtest_reg = xgb.DMatrix(X_test_set, y_test_set, enable_categorical=True)

In [4]:
params = {"objective": "reg:squarederror", "device": "cuda"}
evals = [(dtrain_reg, "train"), (dtest_reg, "validation")]

n = 1000

model = xgb.train(
    params=params,
    dtrain=dtrain_reg,
    num_boost_round=n,
    evals=evals,
    verbose_eval=10,
    early_stopping_rounds=20
)

results = xgb.cv(
    params,
    dtrain_reg,
    num_boost_round=n,
    nfold=5,
    early_stopping_rounds=20
)

[0]	train-rmse:0.12375	validation-rmse:0.12362
[10]	train-rmse:0.05640	validation-rmse:0.05672
[20]	train-rmse:0.05597	validation-rmse:0.05637
[30]	train-rmse:0.05583	validation-rmse:0.05632
[40]	train-rmse:0.05573	validation-rmse:0.05630
[50]	train-rmse:0.05564	validation-rmse:0.05629
[60]	train-rmse:0.05556	validation-rmse:0.05629
[70]	train-rmse:0.05549	validation-rmse:0.05629
[71]	train-rmse:0.05548	validation-rmse:0.05629


In [5]:
from sklearn.metrics import mean_squared_error

preds = model.predict(dtest_reg)

rmse = mean_squared_error(y_test_set, preds, squared=False)
print(f'RMSE of model: {rmse:.4f}')

results.head()

best_rmse = results['test-rmse-mean'].min()
print(best_rmse)

RMSE of model: 0.0563
0.05613199773504156


In [6]:
test_csv = pd.read_csv(os.path.join(input_dir, 'test.csv'))
X_val = test_csv.drop(columns=['id'])

categories = X_val.select_dtypes(exclude=np.number).columns.tolist()
    
for col in categories:
    X_val[col] = X_val[col].astype('category')

dval_reg = xgb.DMatrix(X_val, enable_categorical=True)

preds = model.predict(dval_reg)

ids = test_csv['id']

submission = pd.DataFrame({
    'id': ids,
    'accident_risk': preds
})
submission.to_csv('submission.csv', index=False)